# OPRB: ví dụ định tính cho báo cáo (không cần GPU)

Lấy ảnh, ground truth và output của cả 5 model cho các trang trong `results/analysis/qualitative_picks_sample3000.csv`.
Các trang và đoạn trích được chọn theo quy tắc cố định trên điểm số (xem `src/oprb_eval/qualitative.py`), không chọn tay.

1. Kaggle: bật **Internet**, Accelerator **None** (không tốn quota GPU).
2. **Add Input → Notebook Output** của 4 phiên chứa prediction của cả 5 model:
   `oprb-rescore-0f26a96` (Tesseract, Qwen), `oprb-ppocr-run3000-0f26a96`, `oprb-doctr-run3000-0f26a96`,
   `oprb-lighton-run3000-part2`.
3. **Save Version → Save & Run All**. Kết quả: thư mục `qualitative/` trong tab Output
   (`examples.json`, ảnh cắt `*_crop.png`, ảnh trang `*_page.png` có khung đỏ quanh đoạn trích).

Mất khoảng 10–15 phút, phần lớn là tải 1,75 GB dữ liệu.

In [ ]:
# ===== CẤU HÌNH =====
COMMIT = "main"
SAMPLE = 3000
MODELS = ["tesseract", "ppocrv5_mobile_en", "doctr_fast_parseq", "lightonocr2_1b", "qwen3_vl_2b"]

In [ ]:
import json, os, pathlib, shutil, subprocess, sys

if not pathlib.Path("/kaggle").exists():
    raise SystemExit("Notebook này chạy trên Kaggle.")
FAST, KEEP = pathlib.Path("/kaggle/temp"), pathlib.Path("/kaggle/temp/results")
OUT = pathlib.Path("/kaggle/working/qualitative")
REPO, DATA = FAST / "oprb-eval", FAST / "oprb_test"
for d in (FAST, KEEP, OUT):
    d.mkdir(parents=True, exist_ok=True)
os.environ.update(HF_HOME=str(FAST / "hf"), PYTHONUNBUFFERED="1")

def sh(command, **kwargs):
    print("$", command, flush=True)
    subprocess.run(command, shell=True, check=True, **kwargs)

if not REPO.exists():
    sh(f"git clone -q https://github.com/conthonamanco/oprb-eval.git {REPO}")
sh(f"git fetch -q --tags origin && git checkout -q {COMMIT}", cwd=REPO)
if COMMIT == "main":
    sh("git merge -q --ff-only origin/main", cwd=REPO)
sh("git log -1 --oneline", cwd=REPO)
sh(f'{sys.executable} -m pip install -q -e "{REPO}[data]" pillow')
sh(f"{sys.executable} -m unittest discover -s tests", cwd=REPO)

In [ ]:
# Prediction của 5 model từ Output các phiên run (Add Input -> Notebook Output).
# Fingerprint trong Input phải khớp results/<model>/run_config.json đã commit, tức đúng các prediction đã chấm.
for model in MODELS:
    committed = json.loads((REPO / "results" / model / "run_config.json").read_text())["fingerprint"]
    found = {json.loads(p.read_text())["fingerprint"]
             for p in pathlib.Path("/kaggle/input").glob(f"**/results/{model}/run_config.json")}
    if found != {committed}:
        raise SystemExit(f"{model}: fingerprint trong Input {found} khác bản đã chấm {committed}.")
    files = sorted(pathlib.Path("/kaggle/input").glob(f"**/results/{model}/predictions*.jsonl"))
    if not files:
        raise SystemExit(f"Thiếu prediction của {model}: Add Input -> Notebook Output của phiên chứa nó.")
    target = KEEP / model
    target.mkdir(parents=True, exist_ok=True)
    for f in files:
        # Nhiều phiên có thể chứa cùng một file: file chỉ nối thêm nên bản lớn nhất là bản mới nhất
        dest = target / f.name
        if not dest.exists() or f.stat().st_size > dest.stat().st_size:
            shutil.copy(f, dest)
    config = REPO / "results" / model / "run_config.json"
    shutil.copy(config, target / "run_config.json")
    print(model, [f.name for f in sorted(target.glob("predictions*.jsonl"))])

PY = f"{sys.executable} -m oprb_eval"
sh(f"{PY} download --data {DATA}", cwd=REPO)

In [ ]:
picks = REPO / "results" / "analysis" / f"qualitative_picks_sample{SAMPLE}.csv"
sh(f"{PY} examples --data {DATA} --results {KEEP} --sample {SAMPLE} --picks {picks} --out {OUT}", cwd=REPO)

In [ ]:
# Xem nhanh: mỗi ví dụ một ảnh cắt, GT và output của từng model trên đoạn đó
from IPython.display import Image, display

examples = json.loads((OUT / "examples.json").read_text())
for e in examples:
    print("=" * 100)
    print(e["rule"], e["model"], e["key"], f"value={float(e['value']):.3f}",
          "| words GT", e["reference_words"], "| output", e["output_words"])
    shown = e.get("excerpt")
    if not shown:
        continue
    if "images" in e:
        display(Image(filename=str(OUT / e["images"]["crop"]), width=700))
        print("box scale", e["images"]["box_scale"], "image", e["images"]["image_size"])
    print("GT     :", shown["reference"])
    for model, m in shown["models"].items():
        print(f"{model[:7]:7}: [{m['wer']:.2f}] {m['text']}")
    for field in ("output_tail", "output_head"):
        if field in e:
            print(field, ":", e[field][:600])